https://share.google/aimode/KzpXevFy8vCpGiMCe

# python multiprocessing shared memory

In Python, sharing memory between processes is typically handled through the multiprocessing module, which offers different techniques depending on the complexity and size of your data. Because each process has its own memory space to bypass the Global Interpreter Lock (GIL), direct sharing is not automatic. [1, 2, 3, 4, 5] 

**1. multiprocessing.shared_memory (Modern approach)**

Introduced in Python 3.8, the [multiprocessing.shared_memory](https://docs.python.org/3/library/multiprocessing.shared_memory.html) module allows processes to access the same block of raw bytes without duplication or pickling. [6, 7] 

* SharedMemory: A low-level class for creating or attaching to a raw byte buffer.
* create=True: One process allocates the block.
   * name: Other processes attach to the same block using a unique string identifier.
* ShareableList: A list-like object that stores standard Python types (ints, floats, strings, etc.) in a shared memory block for high-speed access.
* SharedMemoryManager: A higher-level manager that simplifies the lifecycle (creation and cleanup) of shared memory blocks across multiple processes. [3, 8, 9, 10, 11] 

**2. Shared Value and Array (Standard approach)**

For simple data types, use multiprocessing.Value and multiprocessing.Array. These use ctypes to store data in shared memory and include built-in locking to prevent race conditions. [12, 13, 14, 15] 

* Value: Used for a single variable (e.g., an integer or float).
* Array: Used for a one-dimensional array of a specific type (e.g., 'i' for integers). [9, 12, 13, 16, 17] 

**3. Server Process Managers**

The multiprocessing.Manager() creates a server process that hosts Python objects (like lists or dictionaries). Other processes access these objects via proxies. [10, 14, 18, 19, 20] 

* Pros: Supports complex, arbitrary Python objects.
* Cons: Slower than raw shared memory because data must be serialized (pickled) and sent between processes. [14, 16] 

**4. Integration with NumPy**

For large-scale data science, you can map a SharedMemory buffer to a numpy.ndarray. This allows multiple processes to perform computations on the same array without copying it. [9, 10, 16, 21, 22] 

```python
from multiprocessing import shared_memory
import numpy as np

# In process A (Creator)
shm = shared_memory.SharedMemory(create=True, size=1024)
buffer_array = np.ndarray((10,), dtype=np.int64, buffer=shm.buf)

# In process B (Attaching)
existing_shm = shared_memory.SharedMemory(name=shm.name)
shared_array = np.ndarray((10,), dtype=np.int64, buffer=existing_shm.buf)

```

**Key Considerations**

* Synchronization: Shared memory does not inherently prevent processes from writing to the same spot at the same time. You must use multiprocessing.Lock or Semaphores to avoid race conditions.
* Cleanup: You must explicitly call .close() to detach from the memory and .unlink() in exactly one process to destroy the block and prevent memory leaks.
* Data Copies: By default, standard objects passed as arguments to a process are copied (pickled). Use the shared memory tools above to avoid this overhead. [7, 10, 13, 14, 23, 24, 25, 26] 

Would you like to see a complete code example for sharing a dictionary or a large NumPy array?

[1] [https://mingze-gao.com](https://mingze-gao.com/posts/python-shared-memory-in-multiprocessing/#:~:text=Python%203.8%20introduced%20a%20new%20module%20multiprocessing.,costs%20of%20copying%20and%20moving%20things%20around.)
[2] [https://www.credativ.de](https://www.credativ.de/en/blog/hardcore/intenumvalue-a-multiprocessing-safe-enum-value-for-python/#:~:text=When%20it%20comes%20to%20sharing%20data%20across,to%20handle%20sharing%20larger%20complex%20data%20structures.)
[3] [https://docs-python.ru](https://docs-python.ru/standart-library/paket-multiprocessing-python/multiprocessing-shared-memory/)
[4] [https://medium.com](https://medium.com/@aruns89/multiprocessing-in-python-a-guide-to-using-multiple-cpu-cores-f2b3c1bcc83a#:~:text=Multiprocessing%20in%20Python%20creates%20separate%20memory%20spaces,run%20independently%20on%20a%20separate%20CPU%20core.)
[5] [https://layer6.ai](https://layer6.ai/troubleshooting-memory-errors-in-python-parallel-processing/#:~:text=On%20the%20other%20hand%2C%20multi%2Dprocessing%20bypasses%20the,between%20them%20can%20introduce%20overhead%20and%20complexity.)
[6] [https://docs.python.org](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
[7] [https://medium.com](https://medium.com/@cctsai1210/using-sharedmemory-in-python-efficient-data-sharing-techniques-and-applications-ac242beddca7)
[8] [https://github.com](https://github.com/Tinker-Twins/Python-Shared-Memory)
[9] [https://habr.com](https://habr.com/ru/companies/otus/articles/913200/#:~:text=SharedMemory:%20%D1%81%D0%BF%D0%B0%D1%81%D0%B5%D0%BD%D0%B8%D0%B5%20%D0%BE%D1%82%20%D0%BF%D0%B0%D0%B9%D0%BF%D0%BE%D0%B2%20%D0%B8%20pickle%20*,%D0%B2%D0%B8%D0%B4%D1%8F%D1%82%20%D0%BE%D0%B4%D0%B8%D0%BD%20%D0%B8%20%D1%82%D0%BE%D1%82%20%D0%B6%D0%B5%20%D0%B1%D0%BB%D0%BE%D0%BA%20%D0%B1%D0%B0%D0%B9%D1%82%D0%BE%D0%B2)
[10] [https://stackoverflow.com](https://stackoverflow.com/questions/53616039/how-to-use-shared-memory-instead-of-passing-objects-via-pickling-between-multipl)
[11] [https://discuss.python.org](https://discuss.python.org/t/how-do-you-store-objects-in-shared-memory-3-8/8176)
[12] [https://stackoverflow.com](https://stackoverflow.com/questions/14124588/shared-memory-in-multiprocessing)
[13] [https://dev.to](https://dev.to/imsushant12/race-conditions-deadlocks-and-synchronisation-in-python-multiprocessing-o2j#:~:text=Race%20conditions%20happen%20only%20when%20multiple%20processes,shared%20array%20%28multiple%20processes%20can%20update%20elements%29.)
[14] [https://stackoverflow.com](https://stackoverflow.com/questions/10721915/shared-memory-objects-in-multiprocessing)
[15] [https://dev.to](https://dev.to/imsushant12/inter-process-communication-in-python-multiprocessing-with-examples-5ai2)
[16] [https://www.youtube.com](https://www.youtube.com/watch?v=EYMh_CZfeNU&t=2)
[17] [https://www.digitalocean.com](https://www.digitalocean.com/community/tutorials/python-multiprocessing-example)
[18] [https://dev.to](https://dev.to/imsushant12/inter-process-communication-in-python-multiprocessing-with-examples-5ai2#:~:text=multiprocessing.%20Manager%20Manager%20allows%20processes%20to%20share,Pipe%20because%20it%20uses%20proxies%20and%20pickling.)
[19] [https://www.digitalocean.com](https://www.digitalocean.com/community/tutorials/python-multiprocessing-example#:~:text=Using%20multiprocessing.%20Manager%20for%20Complex%20Shared%20Objects,than%20Value%20or%20Array%20for%20tight%20loops.)
[20] [https://docs.python.org](https://docs.python.org/3/tutorial/datastructures.html#:~:text=5.5.%20Dictionaries%20%C2%B6%20Another%20useful%20data%20type,languages%20as%20%E2%80%9Cassociative%20memories%E2%80%9D%20or%20%E2%80%9Cassociative%20arrays%E2%80%9D.)
[21] [https://discuss.python.org](https://discuss.python.org/t/shared-memory-not-working-properly-at-all/23242)
[22] [https://github.com](https://github.com/pytorch/pytorch/issues/70041)
[23] [https://discuss.python.org](https://discuss.python.org/t/should-sharedmemory-close-be-called-explicitly-or-we-should-just-rely-on-del/24281)
[24] [https://indico.ictp.it](https://indico.ictp.it/event/10819/session/9/contribution/38/material/slides/0.pdf)
[25] [https://stackoverflow.com](https://stackoverflow.com/questions/79152934/python3-multiprocessing-sharedmemory-can-not-solve-connecting-to-existing-memory)
[26] [https://habr.com](https://habr.com/ru/articles/789904/)


# My setup is following. There is main processes where main PyTorch model lives. I've created child processes ("spawn" context) which also instantiate the same model in their own memory. Child processes receive task from main process, execute them and report back result. Now I need that just before task execution each child process to sync weights of its model with weights of model from main process. Since weights may occupy lots of memory I want to have a shared memory between processes. So workflow which I need is: 1) main process uploads weights to this shared memory 2) child processes download weights from this shared memory How do I do this?

In PyTorch, the most efficient way to achieve this is using the torch.multiprocessing module, which extends standard Python multiprocessing to handle tensors natively in shared memory. [1, 2] 

Since you are using the spawn context, child processes do not inherit memory from the parent. To avoid manual byte-level management with multiprocessing.shared_memory, you should use the built-in [share_memory_()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.share_memory_.html) method. [3, 4] 

**Recommended Workflow**

   1. Allocate Shared Memory: In the main process, call model.share_memory_() on your primary model. This moves the underlying parameter storages into a shared memory region that persists across process boundaries.
   2. Pass the Model Instance: Pass this shared model instance to the child processes as an argument when you spawn them. Because you called share_memory_(), PyTorch will not pickle the entire model data; instead, it sends a shared handle.
   3. Automatic Synchronization: In the child processes, the parameters of the model instance they received will point to the exact same memory address as the main process's model.
   * Main Process Upload: When you update the main model's weights (e.g., loading a new state_dict), the changes are instantly visible to the children.
      * Child Process Execution: Before a child starts a task, it can simply copy these shared weights into its local model. [1, 5, 6, 7, 8] 
   
**Code Implementation**

```python
import torchimport torch.multiprocessing as mp

def child_worker(task_queue, shared_model, local_model):
    while True:
        task = task_queue.get()
        if task is None: break
        
        # 1. Sync child's local model with the shared weights
        local_model.load_state_dict(shared_model.state_dict())
        
        # 2. Execute task
        result = local_model(task)
        # ... report back result
if __name__ == '__main__':
    mp.set_start_method('spawn')
    
    # Main model (the "source" of weights)
    main_model = MyModel()
    main_model.share_memory_()  # Moves weights to shared memory
    
    # Example child setup
    task_queue = mp.Queue()
    child_model = MyModel() # Child's local instance
    
    p = mp.Process(target=child_worker, args=(task_queue, main_model, child_model))
    p.start()
    
    # Update weights in main process
    # main_model.load_state_dict(...) -> Changes are now visible to p
```

**Important Technical Details**


* CPU Tensors Only: share_memory_() is only for CPU tensors. If you are using GPUs, sharing is handled differently via CUDA IPC, which is automatically active when you pass a CUDA tensor to a child process in spawn mode.
* 
* No Resizing: Tensors in shared memory cannot be resized (e.g., you cannot change the layer dimensions dynamically).
* 
* Locking: If the main process updates weights while a child is reading them, you may get inconsistent weights. Use a mp.Lock() around the load_state_dict call if updates happen frequently during execution. [7, 9, 10, 11] 

Would you like a more detailed example focusing on CUDA/GPU shared memory specifically?

[1] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[2] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[3] [https://stackoverflow.com](https://stackoverflow.com/questions/79666147/pytorch-sharing-cuda-tensors)
[4] [https://github.com](https://github.com/pytorch/pytorch/issues/64932)
[5] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[6] [https://stackoverflow.com](https://stackoverflow.com/questions/78189428/is-there-a-way-to-share-a-pytorch-model-across-multiple-processes-without-using)
[7] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.share_memory_.html)
[8] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[9] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[10] [https://github.com](https://github.com/pytorch/pytorch/issues/70041)
[11] [https://discuss.pytorch.org](https://discuss.pytorch.org/t/sharing-model-between-processes-automatically-allocates-new-memory/96724)


# I have single GPU. Main model is stored in GPU and child models are also stored in GPU.

When working with a single GPU and the spawn context, PyTorch handles shared memory for CUDA tensors automatically through CUDA IPC (Inter-Process Communication). Unlike CPU tensors, calling .share_memory_() on CUDA tensors is a no-op. [1, 2, 3, 4, 5] 

To sync weights without doubling your GPU memory for every child process, follow this workflow:

**1. The Main Process Model**

The main model parameters are already "shared" if you pass them directly. You do not need to call .share_memory_(). Simply ensure your main model is on the GPU before you spawn the children. [4] 

**2. Passing the Model**

When you pass the main model (or its parameters) to the child process via mp.Process(args=...), PyTorch doesn't copy the data. It sends a metadata handle that points to the original GPU memory. [6, 7] 

**3. Syncing without Doubling Memory**

To update your child process weights using the main model's shared memory, use load_state_dict with assign=True (available in recent PyTorch versions) or a manual in-place copy.

```python
import torchimport torch.multiprocessing as mp

def child_worker(shared_model, local_model, task_queue):
    while True:
        task = task_queue.get()
        if task is None: break
        
        # EFFECTIVE SYNC:
        # Option A: In-place copy (avoids temporary memory surge)
        with torch.no_grad():
            for local_param, shared_param in zip(local_model.parameters(), shared_model.parameters()):
                local_param.copy_(shared_param)
        
        # Option B: load_state_dict (cleaner, but can cause small spikes)
        # local_model.load_state_dict(shared_model.state_dict())
        
        # Execute task
        output = local_model(task)
if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)
    
    # 1. Main model on GPU
    main_model = MyModel().cuda()
    
    # 2. Child model also on GPU (separate instance)
    child_model = MyModel().cuda()
    
    # 3. Pass both to child. 'main_model' is passed as a shared handle.
    p = mp.Process(target=child_worker, args=(main_model, child_model, queue))
    p.start()
```

**Critical Rules for GPU Shared Memory**

* Keep the Producer Alive: The main process must keep the original tensors in scope. If the main model is deleted, the memory handles in the child processes become invalid, leading to a segmentation fault.
* Sequential Sync: Since you only have one GPU, both the main model update and the child's copy_() or load_state_dict() share the same hardware. Only one process can truly use the GPU at a time; the driver will serialize these operations.
* Reference Counting: In spawn mode, PyTorch manages reference counts for CUDA IPC. Ensure child processes exit cleanly, or you may leak GPU memory until the main process is restarted. [1, 8, 9, 10] 

Would you like to see how to implement an explicit lock to ensure the main process isn't updating weights while a child is copying them?

[1] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[2] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[3] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.share_memory_.html)
[4] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.share_memory_.html)
[5] [https://discuss.pytorch.org](https://discuss.pytorch.org/t/shared-cuda-tensor-consumes-gpu-memory/134543)
[6] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[7] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[8] [https://discuss.pytorch.org](https://discuss.pytorch.org/t/multiprocessing-cuda-memory/18951)
[9] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[10] [https://github.com](https://github.com/pytorch/pytorch/issues/171039#:~:text=Here%27s%20a%20summary%20of%20the%20current%20PyTorch,allowing%20for%20proper%20release%20of%20GPU%20memory.)


# I see that you've created child_model in main process and passed it to child process. Is this even working? I though child_model should be created by child process in order to keep CUDA happy

You are absolutely correct to be skeptical. While you can pass a model created in the main process to a child, it is generally bad practice with CUDA and the spawn context.

If you create the child_model in the main process and send it, you are effectively sending another "shared handle" back to the same GPU memory. If the child then tries to use it as its own local scratchpad, it might unintentionally interfere with the main process or hit synchronization issues.

The cleanest and most robust way is exactly what you suggested:

   1. Main process owns the "Source of Truth" model.
   2. Child process instantiates its own local_model inside its run method.
   3. Sync occurs by copying weights from the shared "Source of Truth" to the local_model.

**The Corrected "Spawn" Pattern**

```python
import torchimport torch.multiprocessing as mp

def child_worker(shared_model_handle, task_queue):
    # 1. Instantiate the model LOCALLY inside the child process
    # This ensures the CUDA context is clean and owned by this process
    local_model = MyModel().cuda()
    
    while True:
        task = task_queue.get()
        if task is None: break
        
        # 2. Sync weights from the shared handle to local memory
        # This uses CUDA IPC to copy within GPU VRAM (very fast)
        with torch.no_grad():
            for local_p, shared_p in zip(local_model.parameters(), shared_model_handle.parameters()):
                local_p.copy_(shared_p)
        
        # 3. Run inference/task
        result = local_model(task)
if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)
    
    # Create the source model in main process
    main_model = MyModel().cuda()
    
    # We pass 'main_model' to the child. 
    # PyTorch's 'spawn' pickler automatically converts its tensors 
    # into IPC handles rather than copying the data.
    p = mp.Process(target=child_worker, args=(main_model, task_queue))
    p.start()
```

**Why this is better:**

* CUDA Context Isolation: Each child process manages its own CUDA context and local memory properly.
* Memory Efficiency: The local_p.copy_(shared_p) operation happens entirely on the GPU. It’s a device-to-device copy, which is significantly faster than moving anything over the PCIe bus or through CPU RAM.
* State Safety: The local_model can maintain its own specific state (like buffers or BatchNorm running stats) without overwriting the main_model unless you explicitly tell it to.

**One nuance to watch for:**

In spawn mode, the first time a child process accesses a CUDA tensor passed from the parent, it triggers the initialization of the CUDA context in that child. This can cause a small 1-2 second delay during the very first task.

Would you like to know how to handle Optimizer states if the children also need to report back gradients or weight updates?



# That's worked! Thank you! Now the next piece of puzzle. There is a "scoreboard" matrix in main process. This matrix is represented by a tensor in GPU. Each child process must fill its own column in this matrix with results of its own task. How to do this efficiently?

To do this efficiently without moving data back and forth between the CPU and GPU, you can leverage the same CUDA IPC (Inter-Process Communication) mechanism you used for the weights.

Since you already have a shared tensor (the matrix) in the main process, you simply need to pass a reference to that matrix to your child processes. Because it’s a CUDA tensor, they will all point to the same physical memory on the GPU.

**The Strategy**

   1. Create the Matrix in Main: Initialize the full scoreboard matrix on the GPU in the main process.
   2. Pass the Reference: Pass the matrix object to the child processes via your weights_queue or as an initial argument.
   3. In-Place Modification: Each child process identifies its assigned column and writes to it using in-place operations (e.g., matrix[:, column_idx] = results).

**Implementation Example**

```python
def child_worker(index, shared_matrix, task_queue):
    # local_model setup...
    
    while True:
        task = task_queue.get()
        if task is None: break
        
        # ... sync weights and run inference ...
        results = local_model(task) # Assume this returns a 1D tensor
        
        # EFFICIENCY KEY: In-place update to the shared GPU memory
        # This does NOT copy the matrix; it writes directly to the parent's VRAM
        with torch.no_grad():
            shared_matrix[:, index].copy_(results)
if __name__ == '__main__':
    # 1. Create the 'Scoreboard' on GPU
    # Rows = number of metrics, Cols = number of children
    num_children = 4
    scoreboard = torch.zeros((100, num_children)).cuda()
    
    # 2. Spawn children, passing the shared matrix
    processes = []
    for i in range(num_children):
        p = mp.Process(target=child_worker, args=(i, scoreboard, task_queues[i]))
        p.start()
        processes.append(p)
```

**Why this is the most efficient way:**

* Zero CPU-GPU Overhead: The data never leaves the GPU. The child writes directly to the VRAM address owned by the main process.
* No Pickling Issues: Like the state_dict, PyTorch handles the serialization of CUDA tensors by passing a memory handle, which works fine in Jupyter.
* Parallelism: Since each child writes to a unique column, there is no "race condition" in the data itself. However, be aware that the GPU scheduler handles the actual hardware writes, which is very fast.

**Important Caveat: Synchronization**

If your main process needs to read the scoreboard while children are writing to it, you must call torch.cuda.synchronize() in the main process to ensure all kernels launched by the children have finished their copy_() operations before you trust the data.

Would you like to know how to use a multiprocessing.Event to signal the main process exactly when a child has finished its column update?



# I also have another scoreboard as numpy array. How should I share it between main and child processes?

To share a NumPy array between processes efficiently in a spawn context (like Jupyter), you should use [multiprocessing.shared_memory](https://docs.python.org/3/library/multiprocessing.shared_memory.html). Unlike passing a standard array, which creates a copy for every child, this approach allows all processes to point to the same block of RAM. [1, 2, 3] 

**The Strategy for NumPy**

   1. Allocate Shared Memory: Create a SharedMemory block in the main process sized to fit your array's bytes.
   2. Wrap with NumPy: Use np.ndarray with the shared buffer to treat that raw memory as a standard array.
   3. Pass the Name: Only send the unique shm.name (a string) to child processes.
   4. Attach in Child: Children use the name to "attach" to the same memory block. [4, 5, 6, 7] 

**Implementation Example**

```python
import numpy as np
from multiprocessing import shared_memory

# --- MAIN PROCESS ---
# 1. Define shape and type
shape = (100, 4)
dtype = np.float64
nbytes = int(np.prod(shape) * np.dtype(dtype).itemsize)

# 2. Create the shared memory block
shm = shared_memory.SharedMemory(create=True, size=nbytes)

# 3. Create a numpy array backed by this shared memory
scoreboard_np = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
scoreboard_np.fill(0) # Initialize

# Now pass 'shm.name', 'shape', and 'dtype' to your children
# p = mp.Process(target=child_worker, args=(shm.name, shape, dtype, ...))

# --- CHILD PROCESS ---
def child_worker(shm_name, shape, dtype, child_idx):
    # 4. Attach to the existing block using the name
    existing_shm = shared_memory.SharedMemory(name=shm_name)
    
    # 5. Wrap it as a numpy array
    shared_score = np.ndarray(shape, dtype=dtype, buffer=existing_shm.buf)
    
    # 6. Update your column (direct write to parent's RAM)
    shared_score[:, child_idx] = [1.2, 3.4, ...] 
    
    # Clean up reference when done
    existing_shm.close()

```

**Critical Maintenance**

* No Pickling: Never pass the scoreboard_np object itself in the Process args; this triggers a slow copy. Only pass the name string.
* Explicit Cleanup: When your program ends, you must call shm.close() and shm.unlink() in the main process. If you don't unlink, the memory stays allocated in your OS even after the script stops, which can lead to leaks.
* Race Conditions: While writing to unique columns is safe, if the main process reads the array while children are writing, it might see partial updates. Use a multiprocessing.Lock if synchronization is tight. [4, 8, 9, 10] 

Would you like a small helper class to automate the creation and unlinking of these shared arrays?
USER_LOCATION

[1] [https://www.youtube.com](https://www.youtube.com/watch?v=uWbSc84he2Q)
[2] [https://superfastpython.com](https://superfastpython.com/numpy-array-sharedmemory/)
[3] [https://luis-sena.medium.com](https://luis-sena.medium.com/sharing-big-numpy-arrays-across-python-processes-abf0dc2a0ab2#:~:text=SharedMemory%20%28Python%203.8+%29%20SharedMemory%20was%20introduced%20with,don%27t%20want%20to%20use%20any%20external%20libraries.)
[4] [https://docs.python.org](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
[5] [https://docs.python.org](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
[6] [https://www.cs.unb.ca](https://www.cs.unb.ca/~bremner/teaching/cs2613/books/python3-doc/library/multiprocessing.shared_memory.html)
[7] [https://superfastpython.com](https://superfastpython.com/numpy-multiprocessing-share-array-global-variable/)
[8] [https://docs.python.org](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
[9] [https://stackoverflow.com](https://stackoverflow.com/questions/17785275/share-large-read-only-numpy-array-between-multiprocessing-processes)
[10] [https://www.reddit.com](https://www.reddit.com/r/learnpython/comments/4u0xh8/fastest_way_to_share_numpy_arrays_between/)


# Too much fuss. Looks like it's easier to convert to torch.tensor even if its's hosted on CPU. Right?

Spot on. Converting to a CPU torch.Tensor is the "secret weapon" for NumPy-style data in PyTorch multiprocessing.

When you call .share_memory_() on a CPU tensor, PyTorch handles all that low-level shm logic (allocation, file descriptors, and cleanup) for you behind the scenes.

**Why this is the smarter move:**

   1. Zero Copy: Just like the GPU version, it uses shared memory. Passing it to a child process sends a lightweight handle, not the data.
   2. NumPy Compatibility: You can convert it back to a NumPy array in the child process using .numpy() without copying memory. It will still point to the same shared buffer.
   3. Automatic Cleanup: PyTorch manages the shared memory lifecycle. When the tensors go out of scope and the processes exit, the memory is freed automatically. No .unlink() required.

**How to do it:**

```python
import torch
import torch.multiprocessing as mp

# --- MAIN PROCESS ---
# 1. Create as a CPU tensor
scoreboard_cpu = torch.zeros((100, 4))
# 2. Mark for sharing (mandatory for CPU tensors in 'spawn')
scoreboard_cpu.share_memory_()

# Now pass 'scoreboard_cpu' directly to child_worker in args

# --- CHILD PROCESS ---
def child_worker(child_idx, scoreboard_cpu):
    # If you REALLY need NumPy features:
    # This view still points to the shared memory!
    score_np = scoreboard_cpu.numpy()
    
    # Update your column (affects the main process RAM immediately)
    score_np[:, child_idx] = 123.45

```

**Pro-Tip for Efficiency**

If your "task results" are already on the GPU (from your model execution), and you want to put them in this CPU scoreboard:

```python
# Inside child_worker
# gpu_results is [100, 1]
with torch.no_grad():
    scoreboard_cpu[:, child_idx].copy_(gpu_results.cpu())

```


The .copy_() method is highly optimized for these cross-device transfers.

Do you need to implement any Locks to prevent the main process from reading the scoreboard while a child is halfway through a column update?



# I've noticed interesting thing. No matter if I use original multiprocessing module or torch.multiprocessing I've got the same performance. And it looks like that even with original multiprocessing I got efficient memory sharing. How it is possible?

It is possible because torch.multiprocessing is actually just a wrapper around the original multiprocessing module. When you import torch, it automatically registers custom reducers that change how the standard multiprocessing module handles tensors. [1, 2, 3, 4] 

Here is why you see no difference:

* Global Registration: As soon as PyTorch is imported, it modifies the default pickling behavior for tensors across your entire Python environment. Even if you use the standard multiprocessing.Queue or Process, it will use PyTorch's optimized logic to send memory handles instead of copying the actual data.
* CUDA Automatic IPC: For GPU tensors, PyTorch uses CUDA Inter-Process Communication (IPC). This is a low-level driver capability. Whether you use the original module or the PyTorch wrapper, both eventually call the same underlying CUDA API to share the memory pointer.
* Identical Core Logic: For CPU tensors, torch.multiprocessing internally calls the same share_memory_() logic that you would use manually with the standard module to achieve zero-copy sharing. [3, 5, 6, 7, 8, 9, 10] 

In short, torch.multiprocessing exists primarily for convenience—it ensures that share_memory_() is called automatically when tensors are sent through queues, so you don't have to remember to do it yourself. Since you are already managing your state_dict and matrices correctly, the "magic" is happening under the hood regardless of which import you use. [1, 2, 3, 4] 

Would you like to know how to verify if a tensor is actually sharing memory using the .is_shared() method?

[1] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[2] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[3] [https://github.com](https://github.com/pytorch/pytorch/issues/70041#:~:text=Looking%20at%20the%20__init__.py%20of%20the%20main,no%20need%20to%20refactor%20the%20mp%20imports%29.)
[4] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[5] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/multiprocessing.html)
[6] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[7] [https://docs.pytorch.org](https://docs.pytorch.org/docs/stable/notes/multiprocessing.html)
[8] [https://stackoverflow.com](https://stackoverflow.com/questions/79666147/pytorch-sharing-cuda-tensors)
[9] [https://stackoverflow.com](https://stackoverflow.com/questions/56426326/using-tensor-share-memory-vs-multiprocessing-queue-in-pytorch-when-training-m)
[10] [https://stackoverflow.com](https://stackoverflow.com/questions/56426326/using-tensor-share-memory-vs-multiprocessing-queue-in-pytorch-when-training-m)
